# Laboratório de Compressão DNA CROM-IA (V3.6) 🧬
## Setup P/ NVIDIA A100 - Híbrido 50/50

Este Notebook fará automaticamente:
1. Montagem do Google Drive
2. Mineração do Dataset DNA
3. Mesclagem em proporção 50/50 (Mutante vs Humano)
4. Treinamento Unsloth rápido de Qwen2.5
5. Exportação nativa GGUF pro seu Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/CROM-V3.6', exist_ok=True)
print("✅ Drive Montado com Sucesso")

In [ ]:
%%capture
!pip install "unsloth[colab-new]" @ git+https://github.com/unslothai/unsloth.git
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes datasets

### Criação do Algoritmo Dinâmico de CROM Transpiler

In [ ]:
%%writefile gerador_codebook.py

import sys, json, re
from collections import Counter

def extract_ngrams(text, min_len=10, max_len=60):
    tokens = re.split(r'(\n|\s{4})', text)
    ngrams = []
    for window_size in range(3, 15):
        for i in range(len(tokens) - window_size + 1):
            block = "".join(tokens[i:i+window_size])
            if min_len <= len(block) <= max_len:
                ngrams.append(block)
    return ngrams

def build_codebook(dataset_path, output_path="macro_codebook_v4.json"):
    print("\n🔬 Mapeando Entropia...")
    counter = Counter()
    with open(dataset_path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i > 25000: break # Limite de analise pro Ngram nao estourar RAM CPU
            if line.strip():
                try:
                    data = json.loads(line)
                    target = data.get("output", "")
                    if target: counter.update(extract_ngrams(target))
                except: pass
    
    dict_entries = {}
    radix = ['A', 'T', 'C', 'G']
    def get_radix_key(idx):
        if idx < 16: return radix[idx // 4] + radix[idx % 4]
        return radix[(idx // 16) % 4] + radix[(idx // 4) % 4] + radix[idx % 4]
    
    idx = 0
    for block, freq in counter.most_common(5000):
        if freq < 50: break
        if len(block) >= 12 and block.strip():
            key = get_radix_key(idx)
            dict_entries[key] = {"text": block, "freq": freq, "bytes": len(block)}
            idx += 1
            if idx > 200: break

    final_payload = {"version": "4.0", "escape_prefix": "@@", "entries": dict_entries}
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(final_payload, f, ensure_ascii=False, indent=2)
    print(f"✅ Codebook Gerado com {len(dict_entries)} entradas radiciais!")

if __name__ == "__main__":
    build_codebook(sys.argv[1])

In [ ]:
%%writefile transpilador.py
import sys, json, random

def load_codebook(cb_path):
    with open(cb_path, "r", encoding="utf-8") as f: data = json.load(f)
    prefix = data.get("escape_prefix", "@@")
    entries = [(f"{prefix}{k}", v["text"]) for k, v in data["entries"].items()]
    entries.sort(key=lambda x: len(x[1]), reverse=True)
    return entries

def transpile_dataset_50_50(input_path, cb_path, output_path):
    entries = load_codebook(cb_path)
    with open(input_path, "r", encoding="utf-8") as fin, open(output_path, "w", encoding="utf-8") as fout:
        for line in fin:
            if not line.strip(): continue
            try:
                data = json.loads(line)
                # 50% chance de manter intacto (Normal), 50% de virar Mutante
                if random.random() < 0.50:
                    outp = data.get("output", "")
                    for ptr, text_block in entries:
                        if text_block in outp: outp = outp.replace(text_block, ptr)
                    data["output"] = outp
                fout.write(json.dumps(data, ensure_ascii=False) + "\n")
            except: pass
    print("🧬 Híbrido 50/50 concluído e salvo!")

if __name__ == "__main__":
    transpile_dataset_50_50(sys.argv[1], sys.argv[2], sys.argv[3])

### Execução CROM Core (Download Dataset, Gerar Codebook, Transpilar 50/50)

In [ ]:
import urllib.request
print("Baixando Base... (ShareGPT jsonl)")
# Exemplo de mini dataset genérico
url = "https://huggingface.co/datasets/conceptofmind/cot_submix_original/resolve/main/data/train-00000-of-00001.parquet"
# Na pratica, importaremos via libraria datasets.
!pip install datasets

from datasets import load_dataset
print("📥 Carregando Dataset do HuggingFace (OpeAssistant/Alpaca).")
dataset = load_dataset("timdettmers/openassistant-guanaco", split="train")

# Converter para jsonl compativel CROM
dataset.to_json("raw_data.jsonl", orient="records", lines=True)

# ATENÇÃO: Se seu dataset de codigo especifico for outro, mude no comando acima.
!python gerador_codebook.py raw_data.jsonl
!python transpilador.py raw_data.jsonl macro_codebook_v4.json hibrido_5050.jsonl

### Treinamento Mágico e GGUF Export (Unsloth)

In [ ]:
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset

max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit", # 1.5B ideal para Edge ultra-rapido
    max_seq_length = max_seq_length,
    dtype = None, load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 64,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

mutante_dataset = load_dataset("json", data_files="hibrido_5050.jsonl", split="train")

from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = mutante_dataset,
    dataset_text_field = "output",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 4, # Acesso ao A100 permite aumentar
        gradient_accumulation_steps = 4,
        warmup_steps = 50,
        max_steps = 800,
        learning_rate = 2e-5,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 100,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "/content/drive/MyDrive/CROM-V3.6/checkpoints",
    ),
)
print("🚀 Iniciando LoRA (Mistura 50/50)... Trazendo o café.")
trainer.train()

In [ ]:
print("🗜️ Salvando GGUF format 4-bit para Inferno VFS Edge local...")
model.save_pretrained_gguf("/content/drive/MyDrive/CROM-V3.6/Modelos_Finais_GGUF", tokenizer, quantization_method = "q4_k_m")
print("✅ Concluído! Vá ao seu Google Drive e baixe o .gguf finalizado na pasta CROM-V3.6.")